In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Procore — Prime Contract Change Orders (Bronze)
# MAGIC For every project in the company, pulls all Prime Contract Change Orders
# MAGIC and saves raw JSON to the Bronze lakehouse.

# COMMAND ----------

import requests
import json
import time
import datetime
from pyspark.sql import Row

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Auth

# COMMAND ----------

auth_result = mssparkutils.notebook.run("procore_auth", 90)
auth_data = json.loads(auth_result)

ACCESS_TOKEN = auth_data["token"]
COMPANY_ID = auth_data["company_id"]

BASE_URL = "https://api.procore.com"

HEADERS = {
    "Authorization": f"Bearer {ACCESS_TOKEN}",
    "Procore-Company-Id": str(COMPANY_ID),
}

print("Auth successful" if ACCESS_TOKEN else "Auth failed")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Get all projects

# COMMAND ----------

projects_resp = requests.get(
    f"{BASE_URL}/rest/v1.0/projects",
    headers=HEADERS,
    params={"company_id": COMPANY_ID, "serializer_view": "compact", "per_page": 200},
)
projects_resp.raise_for_status()
all_projects = projects_resp.json()
print(f"Found {len(all_projects)} project(s)")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Helper: GET with pagination and 429 retry/backoff

# COMMAND ----------

def get_all_pages(url, params=None, per_page=100, max_retries=5):
    params = dict(params or {})
    params["per_page"] = per_page

    all_results = []
    page = 1
    while True:
        params["page"] = page

        retries = 0
        while True:
            resp = requests.get(url, headers=HEADERS, params=params)

            if resp.status_code == 429:
                retries += 1
                if retries > max_retries:
                    resp.raise_for_status()
                wait_seconds = int(resp.headers.get("Retry-After", 30))
                print(f"  -> 429 rate limited, waiting {wait_seconds}s (retry {retries}/{max_retries})...")
                time.sleep(wait_seconds)
                continue

            if not resp.ok:
                print(f"  -> {resp.status_code} on {resp.url}")
                print(f"  -> body: {resp.text[:500]}")
            resp.raise_for_status()
            break

        batch = resp.json()

        if not isinstance(batch, list):
            raise ValueError(f"Expected a list response, got: {type(batch)} -> {batch}")

        all_results.extend(batch)

        if len(batch) < per_page:
            break
        page += 1

    return all_results

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Pull prime change orders for every project

# COMMAND ----------

all_change_orders = []

for project in all_projects:
    project_id = project["id"]
    project_name = project["name"]

    try:
        change_orders = get_all_pages(
            f"{BASE_URL}/rest/v1.0/projects/{project_id}/prime_change_orders",
            params={"project_id": project_id}
        )
        for co in change_orders:
            co["_project_id"] = project_id
            co["_project_name"] = project_name

        all_change_orders.extend(change_orders)
        print(f"Project {project_id} ({project_name}): {len(change_orders)} change order(s)")

    except requests.HTTPError as e:
        print(f"  [project {project_id}] skipped — {e}")

print(f"\nTotal prime change orders pulled: {len(all_change_orders)}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Save to Bronze lakehouse

# COMMAND ----------

BRONZE_TABLE = "procore_prime_change_orders"

pull_ts = datetime.datetime.utcnow().isoformat()

bronze_rows = [
    Row(
        project_id=co.get("_project_id"),
        project_name=co.get("_project_name"),
        change_order_id=co.get("id"),
        raw_json=json.dumps(co),
        pulled_at=pull_ts,
    )
    for co in all_change_orders
]

if bronze_rows:
    bronze_df = spark.createDataFrame(bronze_rows)
    bronze_df.write.format("delta").mode("append").saveAsTable(BRONZE_TABLE)
    print(f"Wrote {bronze_df.count()} row(s) to {BRONZE_TABLE}")
else:
    print("No prime change orders pulled this run — nothing written to bronze.")

StatementMeta(, f0517877-c1f2-47a4-937e-20d0653f9f73, 4, Finished, Available, Finished, True)

Auth successful
Found 18 project(s)
Project 562949955001573 (25-016 - 1100 Fulton Street): 35 change order(s)
Project 562949954833574 (24-011  - 11 ESSEX ST): 97 change order(s)
Project 562949955118102 (25-018 - 337A & 337B West Broadway Rehabilitaion Work): 10 change order(s)
Project 562949955225798 (25-021 - 360 Lexington 8th & 20th Floor): 50 change order(s)
Project 562949955257421 (25-020 - 549 Munroe Av): 0 change order(s)
Project 562949955064640 (25-017 - 64 MET OVAL PSC + 1410 MET STOREROOM): 60 change order(s)
Project 562949955318524 (26-023 - Boys & Girls Club): 0 change order(s)
Project 562949955286476 (26-022 - EMBANKMENT PHASE II): 5 change order(s)
Project 562949955375634 (26-026 - Embankment Phase III): 2 change order(s)
Project 562949954973730 (25-014 - Embankment + Revetment Apartments 270 & 310 10th Street NJ): 53 change order(s)
Project 562949954973684 (25-013 - Lillipvt 45 Renwick St): 6 change order(s)
Project 562949954971827 (25-012 - PCNA 711 11TH AVE): 22 change 